# Weekly stock prices from 2020

In [21]:
# todo
# calculate 4 sets of 4 weekly log returns staggered a week apart
# each asset is going to appear in each set
# start portfolio as of 2020-01-01, end portfolio as of
# 323 weeks back of september 2, 2026
# add australian dollar (AUD) as a currency
# get the sector for each stock

# 11 secotor etf returns, returns of spy
# take each returns to each sector and regress them on the market to get the sector beta
# get the residual returns from that regression. This will result in residual sector returns

# timewieght the returns by 0.986 ** time period 

# stagger them by 22,6,2020
# 29, 6, 20
# 6, 7, 20
# 13, 7, 20

# Load General Data

In [22]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import pycountry
from babel.numbers import get_territory_currencies


DATA_DIR = Path('../data')
companies = pd.read_excel(DATA_DIR / 'QPM Student Stock Picks.xlsx', header=1)
companies = companies[['TICKER', '      Company Name']]
companies = companies[companies['TICKER'] != 'SSNLF']

companies

,TICKER,Company Name
0,VST,Vistra Corp.
1,VIST,"Vista Energy, S.A.B. de C.V."
2,AEP,"American Electric Power Company, Inc."
3,ETR,Entergy Corporation
4,AEP,"American Electric Power Company, Inc."
5,DIS,The Walt Disney Company
6,CMG,"Chipotle Mexican Grill, Inc."
7,AMZN,"Amazon.com, Inc."
8,FIX,"Comfort Systems USA, Inc."
9,STX,Seagate Technology Holdings plc


In [23]:
companies['yahoo_ticker'] = companies['TICKER'].replace({'BRK.B': 'BRK-B'})
companies['company_name'] = companies['      Company Name']
companies = companies.drop(['TICKER', '      Company Name'], axis = 1)

start_date = '2020-6-22'  # First Monday of Start june 22 2020.
end_date = '2026-9-1' # end on august 31, 2026


In [24]:
companies

,yahoo_ticker,company_name
0,VST,Vistra Corp.
1,VIST,"Vista Energy, S.A.B. de C.V."
2,AEP,"American Electric Power Company, Inc."
3,ETR,Entergy Corporation
4,AEP,"American Electric Power Company, Inc."
5,DIS,The Walt Disney Company
6,CMG,"Chipotle Mexican Grill, Inc."
7,AMZN,"Amazon.com, Inc."
8,FIX,"Comfort Systems USA, Inc."
9,STX,Seagate Technology Holdings plc


# Add countries and sector etfs

In [25]:

sector_etfs = [
    ('XLC', 'Communication Services'),
    ('XLY', 'Consumer Discretionary'),
    ('XLP', 'Consumer Staples'),
    ('XLE', 'Energy'),
    ('XLF', 'Financials'),
    ('XLV', 'Health Care'),
    ('XLI', 'Industrials'),
    ('XLB', 'Materials'),
    ('XLRE', 'Real Estate'),
    ('XLK', 'Technology'),
    ('XLU', 'Utilities'),
]

In [26]:
new_rows = pd.DataFrame(
    [
        ("SPY", "S&P 500 ETF", "United States"),
        *[
            (ticker, sector_name, "United States")
            for ticker, sector_name in sector_etfs
        ],
    ],
    columns=["yahoo_ticker", "company_name", "country"],
)

companies["country"] = pd.NA
companies = pd.concat([companies, new_rows], ignore_index=True)

In [27]:
companies

,yahoo_ticker,company_name,country
0,VST,Vistra Corp.,<NA>
1,VIST,"Vista Energy, S.A.B. de C.V.",<NA>
2,AEP,"American Electric Power Company, Inc.",<NA>
3,ETR,Entergy Corporation,<NA>
4,AEP,"American Electric Power Company, Inc.",<NA>
...,...,...,...
56,XLI,Industrials,United States
57,XLB,Materials,United States
58,XLRE,Real Estate,United States
59,XLK,Technology,United States


In [28]:
# get the currencies and sectors for each company

def get_country(ticker):
    try:
        return yf.Ticker(ticker).info.get('country')
    except:
        return None
    
def get_sector(ticker):
    try:
        return yf.Ticker(ticker).info.get('sector')
    except:
        return None

countries = companies['yahoo_ticker'].apply(get_country)
sectors = companies['yahoo_ticker'].apply(get_sector)

In [29]:
companies['country'] = countries
companies['sector'] = sectors
companies

,yahoo_ticker,company_name,country,sector
0,VST,Vistra Corp.,United States,Utilities
1,VIST,"Vista Energy, S.A.B. de C.V.",Mexico,Energy
2,AEP,"American Electric Power Company, Inc.",United States,Utilities
3,ETR,Entergy Corporation,United States,Utilities
4,AEP,"American Electric Power Company, Inc.",United States,Utilities
...,...,...,...,...
56,XLI,Industrials,NaN,NaN
57,XLB,Materials,NaN,NaN
58,XLRE,Real Estate,NaN,NaN
59,XLK,Technology,NaN,NaN


In [30]:
# replace occurences: 
#  Consumer Cylical = Consumer Discretionary
#  Consumer Defensive = Consumer Staples
#  Basic Materials = Matierals
#  Healthcare = Health Care
#. Financial Services = Financials

companies.loc[companies['sector'].isin(['Consumer Cyclical']),
              ['sector']] = ['Consumer Discretionary']

companies.loc[companies['sector'].isin(['Consumer Defensive']),
              ['sector']] = ['Consumer Staples']

companies.loc[companies['sector'].isin(['Basic Materials']),
              ['sector']] = ['Materials']

companies.loc[companies['sector'].isin(['Healthcare']),
              ['sector']] = ['Health Care']

companies.loc[companies['sector'].isin(['Financial Services']),
              ['sector']] = ['Financials']

# Get local currencies

In [31]:
local_currencies = []

for idx, row in companies.iterrows():
    country = row['country']
    try:
        country_code = pycountry.countries.lookup(country).alpha_2
        currency = get_territory_currencies(country_code)[0]
        local_currencies.append(currency)
    except:
        local_currencies.append(None)

In [32]:
companies['currency'] = local_currencies
companies

,yahoo_ticker,company_name,country,sector,currency
0,VST,Vistra Corp.,United States,Utilities,USD
1,VIST,"Vista Energy, S.A.B. de C.V.",Mexico,Energy,MXN
2,AEP,"American Electric Power Company, Inc.",United States,Utilities,USD
3,ETR,Entergy Corporation,United States,Utilities,USD
4,AEP,"American Electric Power Company, Inc.",United States,Utilities,USD
...,...,...,...,...,...
56,XLI,Industrials,NaN,NaN,NaN
57,XLB,Materials,NaN,NaN,NaN
58,XLRE,Real Estate,NaN,NaN,NaN
59,XLK,Technology,NaN,NaN,NaN


In [33]:
# Exchange rates in USD per unit of foreign currency
for curr in dict.fromkeys(companies['currency'].unique()):
    if curr is np.nan or curr == 'USD':
        continue
    fx_ticker = f'{curr}USD=X'
    if fx_ticker not in companies['yahoo_ticker'].values:
        companies.loc[len(companies)] = {
            'ticker': fx_ticker,
            'yahoo_ticker': fx_ticker,
            'company_name': f'{curr}/USD exchange rate',
        }

# add AUD since prof wanted it
companies.loc[len(companies)] = {
            'ticker': 'AUDUSD=X',
            'yahoo_ticker': 'AUDUSD=X',
            'company_name': 'AUD/USD exchange rate',
        }

In [34]:
companies

,yahoo_ticker,company_name,country,sector,currency
0,VST,Vistra Corp.,United States,Utilities,USD
1,VIST,"Vista Energy, S.A.B. de C.V.",Mexico,Energy,MXN
2,AEP,"American Electric Power Company, Inc.",United States,Utilities,USD
3,ETR,Entergy Corporation,United States,Utilities,USD
4,AEP,"American Electric Power Company, Inc.",United States,Utilities,USD
...,...,...,...,...,...
62,SGDUSD=X,SGD/USD exchange rate,NaN,NaN,NaN
63,CADUSD=X,CAD/USD exchange rate,NaN,NaN,NaN
64,DKKUSD=X,DKK/USD exchange rate,NaN,NaN,NaN
65,EURUSD=X,EUR/USD exchange rate,NaN,NaN,NaN


In [35]:
price_frames = []

download_companies = (
    companies
    .dropna(subset=["yahoo_ticker"])
    .drop_duplicates(subset=["yahoo_ticker"])
)

for company in download_companies.itertuples(index=False):
    history = yf.download(
        company.yahoo_ticker,
        start=start_date,
        end=end_date,
        interval='1wk',
        auto_adjust=False,
        progress=False,
        multi_level_index=False,
    )
    history.index.name = 'date'
    history = history.reset_index().rename(columns={
        'Open': 'open', 'High': 'high', 'Low': 'low',
        'Close': 'close', 'Adj Close': 'adj_close', 'Volume': 'volume',
    })
    history['ticker'] = company.yahoo_ticker
    history['company_name'] = company.company_name
    price_frames.append(history)

prices = pd.concat(price_frames, ignore_index=True)
prices = prices[['date', 'ticker', 'company_name', 'open', 'high', 'low', 'close', 'adj_close', 'volume']]
prices.to_csv(DATA_DIR / 'historical_prices.csv', index=False)


In [36]:
ticker_order = (
    companies["yahoo_ticker"]
    .dropna()
    .drop_duplicates()
)

adjusted_close = (
    prices
    .pivot(index="date", columns="ticker", values="adj_close")
    .reindex(columns=ticker_order)
    .rename(columns={
        "MXNUSD=X": "MXN/USD",
        "CADUSD=X": "CAD/USD",
        "DKKUSD=X": "DKK/USD",
        "EURUSD=X": "EUR/USD",
        "AUDUSD=X": "AUD/USD",
        "KRWUSD=X": "KRW/USD",
        "SGDUSD=X": "SGD/USD"
    })
)

adjusted_close = (
    adjusted_close
    .rename_axis(columns=None)
    .reset_index()
)


adjusted_close


,date,VST,VIST,AEP,ETR,DIS,CMG,AMZN,FIX,STX,...,XLB,XLRE,XLK,XLU,MXN/USD,SGD/USD,CAD/USD,DKK/USD,EUR/USD,AUD/USD
0,2020-06-22,15.996824,3.060000,62.459698,36.770145,105.779343,20.663000,134.643494,37.064911,38.835629,...,24.118837,27.317120,48.430229,22.817087,0.043386,0.717824,0.730151,0.150559,1.121989,0.688819
1,2020-06-29,16.799751,3.130000,66.079147,38.856178,108.765594,21.129000,144.514999,38.861813,39.141563,...,25.464434,28.863993,50.209278,23.941809,0.044700,0.717201,0.745523,0.150950,1.125000,0.691920
2,2020-07-06,16.649755,3.210000,67.384407,39.373665,115.707664,22.360800,160.000000,37.764252,39.174633,...,25.668043,28.345663,51.549538,23.888050,0.044565,0.719114,0.736052,0.151720,1.130000,0.695200
3,2020-07-13,17.311512,3.170000,70.074966,40.292313,115.038666,22.724400,148.098495,40.687870,39.910492,...,27.071173,28.361858,50.953346,24.897001,0.044382,0.719600,0.736312,0.153497,1.143100,0.697800
4,2020-07-20,16.658579,3.620000,70.507385,41.182884,114.030319,22.605200,150.445496,39.901112,39.720325,...,27.208391,28.183687,50.175888,24.925949,0.044900,0.723327,0.745545,0.156654,1.165660,0.710010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,2026-08-03,140.360123,65.959999,124.779999,106.543976,104.910004,32.790001,274.480011,1693.672974,812.109009,...,52.617855,44.604549,187.750900,43.291679,0.058403,0.782779,0.716743,0.154619,1.156337,0.705700
320,2026-08-10,147.887817,68.309998,125.599998,107.839996,106.849998,33.500000,262.649994,1774.060059,972.660339,...,52.299320,44.892132,189.788513,43.986572,0.058794,0.781800,0.720773,0.154796,1.156872,0.706500
321,2026-08-17,135.987305,69.580002,120.940002,104.620003,107.779999,36.900002,258.630005,1655.609985,849.319153,...,53.294739,44.703716,183.096329,42.457813,0.059162,0.788022,0.726216,0.156238,1.168907,0.717206
322,2026-08-24,136.865845,70.139999,122.309998,105.750000,108.099998,38.029999,266.429993,1518.729980,829.095398,...,52.936386,44.108723,185.473557,42.418102,0.058753,0.785546,0.719062,0.154969,1.158749,0.718100


In [37]:
na_counts = adjusted_close.isna().sum()

# Show only columns containing at least one NaN
na_counts = na_counts[na_counts > 0].sort_values(ascending=False)

print(na_counts)

DOCN    39
UPST    25
dtype: int64


In [38]:
companies.loc[companies['yahoo_ticker'].isin(['MXNUSD=X']),
              ['yahoo_ticker']] = ['MXN/USD']

companies.loc[companies['yahoo_ticker'].isin(['CADUSD=X']),
              ['yahoo_ticker']] = ['CAD/USD']

companies.loc[companies['yahoo_ticker'].isin(['DKKUSD=X']),
              ['yahoo_ticker']] = ['DKK/USD']

companies.loc[companies['yahoo_ticker'].isin(['EURUSD=X']),
              ['yahoo_ticker']] = ['EUR/USD']

companies.loc[companies['yahoo_ticker'].isin(['AUDUSD=X']),
              ['yahoo_ticker']] = ['AUD/USD']

companies.loc[companies['yahoo_ticker'].isin(['KRWUSD=X']),
              ['yahoo_ticker']] = ['KRW/USD']

companies.loc[companies['yahoo_ticker'].isin(['SGDUSD=X']),
              ['yahoo_ticker']] = ['SGD/USD']


In [39]:
companies.rename(columns = {'yahoo_ticker': 'ticker'}, inplace=True)

In [40]:
with pd.ExcelWriter(DATA_DIR / "PMData.xlsx", engine="openpyxl") as writer:
    adjusted_close.to_excel(writer, sheet_name="Adjusted Close", index = False)
    companies.to_excel(writer, sheet_name="Company Meta Data", index = False)
